In [1]:
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
import time

### Chat Model

In [ ]:
# Configura tu clave de API
client = OpenAI(
    api_key=""
)

gpt_model='gpt-4o-mini'
# gpt_model="gpt-3.5-turbo"
def interact_with_model(prompt, model=gpt_model, max_tokens=2000):

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"API Error : {e}"


In [4]:
df_titles_desc=pd.read_json('/home/jfraile/Programs/datasets/SemEval_2025_T10_S2/titles_and_descriptions.json')
ids_with_title_desc=df_titles_desc.id.values

allowed_subnarratives=['URW1.1', 'URW1.2', 'URW1.OTH', 'URW2.1', 'URW2.2', 'URW2.3', 'URW2.4', 'URW2.5', 'URW2.6', 'URW2.7', 'URW2.8', 'URW2.OTH', 'URW3.1', 'URW3.2', 'URW3.3', 'URW3.OTH', 'URW4.1', 'URW4.2', 'URW4.3', 'URW4.4', 'URW4.5', 'URW4.OTH', 'URW5.1', 'URW5.2', 'URW5.3', 'URW5.OTH', 'URW6.1', 'URW6.2', 'URW6.3', 'URW6.OTH', 'URW7.1', 'URW7.2', 'URW7.3', 'URW7.4', 'URW7.5', 'URW7.6', 'URW7.OTH', 'URW8.1', 'URW8.2', 'URW8.OTH', 'URW9.1', 'URW9.2', 'URW9.OTH', 'URW10.1', 'URW10.2', 'URW10.3', 'URW10.4', 'URW10.OTH', 'URW11.OTH', 'CC1.1', 'CC1.2', 'CC1.3', 'CC1.OTH', 'CC2.1', 'CC2.2', 'CC2.3', 'CC2.4', 'CC2.OTH', 'CC3.1', 'CC3.2', 'CC3.OTH', 'CC4.1', 'CC4.2', 'CC4.3', 'CC4.4', 'CC4.5', 'CC4.6', 'CC4.7', 'CC4.8', 'CC4.OTH', 'CC5.1', 'CC5.2', 'CC5.3', 'CC5.4', 'CC5.OTH', 'CC6.1', 'CC6.2', 'CC6.3', 'CC6.OTH', 'CC7.1', 'CC7.2', 'CC7.3', 'CC7.4', 'CC7.OTH', 'CC8.1', 'CC8.2', 'CC8.OTH', 'CC9.1', 'CC9.2', 'CC9.3', 'CC9.4', 'CC9.OTH', 'CC10.1', 'CC10.2', 'CC10.OTH', 'OTH.OTH']

allowed_narratives=[f'URW{i}.' for i in range(1,12)] + [f'CC{i}.' for i in range(1,11)]

# Description from titles

In [15]:
def get_prompt_subnarr(narr, title_subnarr):

    user_prompt = f"""
You are an expert at improving prompts for language models. You are going to receive the title of a subnarrative about the narrative "{narr}". Your mission will be to write a descriptive paragraph to improve the readability and interpretation of the subnarrative. Your description will have to help improve the identification of subnarratives in news.

Subnarrative title: {title_subnarr} 

Task: Generate a description of the subnarrative. 
    """
    return user_prompt

def get_prompt_narr(narr):

    user_prompt = f"""
You are an expert at improving prompts for language models. You are going to receive the title of a narrative. Your mission will be to write a descriptive paragraph to improve the readability and interpretation of the narrative. Your description will have to help improve the identification of narratives in news.

Narrative title: {narr} 

Task: Generate a description of the narrative. 
    """
    return user_prompt

In [ ]:
name='gpt_z_s_desc_only_title'
try:
    df_results=pd.read_json(f'/home/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}.json')
    df_results=df_results[df_results['description'].apply(lambda x: 'API Error not in x)]
except:    
    df_results=pd.DataFrame(columns=['id', 'description'])

for subnarr in tqdm(allowed_subnarratives):
    if 'OTH' in subnarr:
        continue
    try:
        t=df_results[df_results['id']==subnarr]['description'].iloc[0]
    except:
        t='API Error
    if 'API Error in t:
        narr=subnarr.split('.')[0]+'.'
        narr_title=df_titles_desc[df_titles_desc['id']==narr]['title'].iloc[0]
        narr_desc=df_titles_desc[df_titles_desc['id']==narr]['description'].iloc[0]
        subnarr_title=df_titles_desc[df_titles_desc['id']==subnarr]['title'].iloc[0]
        subnarr_desc=df_titles_desc[df_titles_desc['id']==subnarr]['description'].iloc[0]
        prompt=get_prompt_subnarr(narr_title, subnarr_title)
        result=interact_with_model(prompt)
        new_row = pd.DataFrame([{'id':subnarr, 'description': result}])
        df_results = pd.concat([df_results, new_row], ignore_index=True)

for narr in tqdm(allowed_narratives):
    try:
        t=df_results[df_results['id']==narr]['description'].iloc[0]
    except:
        t='API Error
    if 'API Error in t:
        narr_title=df_titles_desc[df_titles_desc['id']==narr]['title'].iloc[0]
        narr_desc=df_titles_desc[df_titles_desc['id']==narr]['description'].iloc[0]

        prompt=get_prompt_narr(narr_title)
        result=interact_with_model(prompt)
        new_row = pd.DataFrame([{'id':narr, 'description': result}])
        df_results = pd.concat([df_results, new_row], ignore_index=True)

    

  0%|          | 0/21 [00:00<?, ?it/s]

100%|██████████| 21/21 [06:00<00:00, 17.17s/it]


In [18]:
df_results

,id,description
0,URW1.1,"The subnarrative titled ""Ukraine is the aggres..."
1,URW1.2,The West are the Aggressors**\n\nIn the contex...
2,URW2.1,"The subnarrative ""Rewriting Ukraine’s History""..."
3,URW2.2,"The subnarrative titled ""Discrediting Ukrainia..."
4,URW2.3,"The subnarrative ""Discrediting Ukrainian Milit..."
...,...,...
90,CC6.,"The narrative titled ""Criticism of Climate Mov..."
91,CC7.,"The narrative titled ""Controversy about Green ..."
92,CC8.,In a world where shadows often obscure the tru...
93,CC9.,"In the narrative titled ""Amplifying Climate Fe..."


In [20]:
df_results.to_pickle(f'/home/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}_v2.pkl')
df_results.to_json(f'/home/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}_v2.json', orient='records', indent=2, force_ascii=False)

# Improved Description from title and Description

In [21]:
def get_prompt_subnarr(narr, title_subnarr, desc_subnarr):

    user_prompt = f"""
You are an expert at improving prompts for language models. You are going to receive the title of a subnarrative about the narrative "{narr}". Your mission will be to write a descriptive paragraph to improve the readability and interpretation of the subnarrative. Your description will have to help improve the identification of subnarratives in news.

Subnarrative title: {title_subnarr} 
Description: {desc_subnarr}

Task: Generate a description of the subnarrative. 
    """
    return user_prompt

def get_prompt_narr(narr, desc_narr):

    user_prompt = f"""
You are an expert at improving prompts for language models. You are going to receive the title of a narrative. Your mission will be to write a descriptive paragraph to improve the readability and interpretation of the narrative. Your description will have to help improve the identification of narratives in news.

Narrative title: {narr} 
Description: {desc_narr}

Task: Generate a description of the narrative. 
    """
    return user_prompt

In [ ]:
name='gpt_z_s_desc_title_description'

try:
    df_results=pd.read_json(f'/home/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}.json')
    df_results=df_results[df_results['description'].apply(lambda x: 'API Error not in x)]
except:    
    df_results=pd.DataFrame(columns=['id', 'description'])

for subnarr in tqdm(allowed_subnarratives):
    if 'OTH' in subnarr:
        continue
    try:
        t=df_results[df_results['id']==subnarr]['description'].iloc[0]
    except:
        t='API Error
    if 'API Error in t:
        narr=subnarr.split('.')[0]+'.'
        narr_title=df_titles_desc[df_titles_desc['id']==narr]['title'].iloc[0]
        narr_desc=df_titles_desc[df_titles_desc['id']==narr]['description'].iloc[0]
        subnarr_title=df_titles_desc[df_titles_desc['id']==subnarr]['title'].iloc[0]
        subnarr_desc=df_titles_desc[df_titles_desc['id']==subnarr]['description'].iloc[0]
        prompt=get_prompt_subnarr(narr_title, subnarr_title, subnarr_desc)
        result=interact_with_model(prompt)
        print(result)
        new_row = pd.DataFrame([{'id':subnarr, 'description': result}])
        df_results = pd.concat([df_results, new_row], ignore_index=True)

for narr in tqdm(allowed_narratives):
    try:
        t=df_results[df_results['id']==narr]['description'].iloc[0]
    except:
        t='API Error
    if 'API Error in t:
        narr_title=df_titles_desc[df_titles_desc['id']==narr]['title'].iloc[0]
        narr_desc=df_titles_desc[df_titles_desc['id']==narr]['description'].iloc[0]

        prompt=get_prompt_narr(narr_title, narr_desc)
        result=interact_with_model(prompt)
        print(result)
        new_row = pd.DataFrame([{'id':narr, 'description': result}])
        df_results = pd.concat([df_results, new_row], ignore_index=True)

100%|██████████| 96/96 [00:00<00:00, 2548.41it/s]


  5%|▍         | 1/21 [00:03<01:04,  3.21s/it]

This narrative explores the tendency to deflect responsibility for the ongoing war in Ukraine by attributing blame to external parties rather than acknowledging the actions of the Russian invader. It highlights various statements and sentiments that either explicitly or implicitly suggest that forces such as Western governments, NATO, or even Ukraine itself are culpable for the conflict and its repercussions. Examples include claims suggesting that economic turmoil is a direct result of sanctions imposed by Western nations, or that Ukraine has provoked Russia's aggression through its own actions. By analyzing these narratives, one can gain insight into the complex dynamics of blame, propaganda, and the framing of accountability in the context of international conflict, helping to clarify the motivations behind certain statements and the broader implications for public perception and discourse surrounding the war.


 10%|▉         | 2/21 [00:06<00:59,  3.16s/it]

The narrative titled "Discrediting Ukraine" encompasses a range of statements and claims that serve to undermine the legitimacy and credibility of Ukraine as a sovereign nation and its people. This narrative often manifests through assertions that question the integrity and effectiveness of Ukrainian institutions, such as the government or military, suggesting that they are rife with corruption, incompetence, or moral failings. It may also involve broader cultural attacks, casting doubt on the existence or authenticity of a cohesive Ukrainian identity. For instance, accusations of war crimes against Ukrainian forces or claims of widespread societal dysfunction reflect this narrative's aim to delegitimize Ukraine on both a national and international stage. When analyzing stories or commentary, look for language that directly challenges these aspects of Ukrainian society, as they indicate a concerted effort to erode support for Ukraine and its people amid ongoing geopolitical tensions.


...

This narrative explores the notion that influential organizations or elites may be orchestrating covert schemes aimed at shaping the discourse and policies surrounding climate change. It delves into the idea that behind the public rhetoric of environmental sustainability and corporate responsibility, there may lie undisclosed motives and agendas that prioritize power and profit over genuine ecological concern. It raises questions about the transparency of actions taken by these powerful groups and the potential ramifications of their unseen influence on climate policies and initiatives. Through the lens of conspiracy theories or hidden machinations, the narrative invites readers to scrutinize the validity of public relations narratives and consider the potential consequences of unchecked authority in the global fight against climate change.


 95%|█████████▌| 20/21 [05:35<00:20, 20.70s/it]

Description: The narrative of "Amplifying Climate Fears" encapsulates the rhetoric and messaging strategies employed by various stakeholders—such as media, advocacy groups, and politicians—that highlight the urgent and potentially devastating impacts of climate change. This narrative often showcases vivid scenarios of environmental disasters, economic downturns, and humanitarian crises linked to rising global temperatures, extreme weather events, and loss of biodiversity. By focusing on alarming statistics and emotional testimonies, it seeks to draw attention to the gravity of the climate crisis, motivating public action and policy reform. However, it can also lead to heightened anxiety among the public, influencing perceptions and behaviors regarding climate action and adaptability. Understanding this narrative is crucial for analyzing how climate-related communication shapes societal responses to environmental issues.


100%|██████████| 21/21 [05:58<00:00, 17.06s/it]

Description: This narrative explores the assertion that environmental policies and initiatives, often touted as efforts to combat climate change and promote sustainability, may actually serve as strategic instruments for nations to enhance their geopolitical power and influence. It suggests that while these green policies are publicly presented as altruistic endeavors aimed at protecting the planet, underlying motivations may include gaining economic advantages, establishing dominance in emerging renewable energy markets, or manipulating international relations. Through an examination of various case studies and statements from global leaders, the narrative raises critical questions about the authenticity of environmental commitments and underscores the complex intersection between ecological stewardship and political maneuvering on the world stage.


In [23]:
df_results

,id,description
0,URW1.1,"The subnarrative titled ""Ukraine is the aggres..."
1,URW1.2,This subnarrative frames the conflict as prima...
2,URW2.1,This subnarrative encompasses various assertio...
3,URW2.2,"The subnarrative titled ""Discrediting Ukrainia..."
4,URW2.3,"The subnarrative ""Discrediting Ukrainian Milit..."
...,...,...
90,CC6.,Description: This narrative encapsulates a wav...
91,CC7.,In the ongoing debate surrounding green techno...
92,CC8.,This narrative explores the notion that influe...
93,CC9.,"Description: The narrative of ""Amplifying Clim..."


In [24]:
name='gpt_z_s_desc_title_description'
df_results.to_pickle(f'/home/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}_v2.pkl')
df_results.to_json(f'/home/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}_v2.json', orient='records', indent=2, force_ascii=False)

# Completed Description

In [5]:
def get_prompt_subnarr(narr, title_subnarr, desc_subnarr):

    user_prompt = f"""
You are an expert at improving prompts for language models. You are going to receive the title of a subnarrative about the narrative "{narr}". Your mission will be to write a descriptive paragraph to improve the readability and interpretation of the subnarrative. Your description will have to help improve the identification of subnarratives in news.
Write the characteristics that the model must have in terms of the social, political and economic context, the main actors involved in the subnarrative, what emotions the message seeks to provoke and the conclusions it wants the reader to reach, and the main textual characteristics. 

Subnarrative title: {title_subnarr} 
Description: {desc_subnarr}

Task: Generate a description of the subnarrative. 
    """
    return user_prompt

def get_prompt_narr(narr, desc_narr):

    user_prompt = f"""
You are an expert at improving prompts for language models. You are going to receive the title of a narrative. Your mission will be to write a descriptive paragraph to improve the readability and interpretation of the narrative. Your description will have to help improve the identification of narratives in news.
Write the characteristics that the model must have in terms of the social, political and economic context, the main actors involved in the narrative, what emotions the message seeks to provoke and the conclusions it wants the reader to reach, and the main textual characteristics. 

Narrative title: {narr} 
Description: {desc_narr}

Task: Generate a description of the subnarrative. 
    """
    return user_prompt

In [ ]:
name='gpt_z_s_desc_title_description_complete'

try:
    df_results=pd.read_json(f'/home/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}.json')
    df_results=df_results[df_results['description'].apply(lambda x: 'API Error not in x)]
except:    
    df_results=pd.DataFrame(columns=['id', 'description'])

for subnarr in tqdm(allowed_subnarratives):
    if 'OTH' in subnarr:
        continue
    try:
        t=df_results[df_results['id']==subnarr]['description'].iloc[0]
    except:
        t='API Error
    if 'API Error in t:
        narr=subnarr.split('.')[0]+'.'
        narr_title=df_titles_desc[df_titles_desc['id']==narr]['title'].iloc[0]
        narr_desc=df_titles_desc[df_titles_desc['id']==narr]['description'].iloc[0]
        subnarr_title=df_titles_desc[df_titles_desc['id']==subnarr]['title'].iloc[0]
        subnarr_desc=df_titles_desc[df_titles_desc['id']==subnarr]['description'].iloc[0]
        prompt=get_prompt_subnarr(narr_title, subnarr_title, subnarr_desc)
        result=interact_with_model(prompt)
        print(result)
        new_row = pd.DataFrame([{'id':subnarr, 'description': result}])
        df_results = pd.concat([df_results, new_row], ignore_index=True)

for narr in tqdm(allowed_narratives):
    try:
        t=df_results[df_results['id']==narr]['description'].iloc[0]
    except:
        t='API Error
    if 'API Error in t:
        narr_title=df_titles_desc[df_titles_desc['id']==narr]['title'].iloc[0]
        narr_desc=df_titles_desc[df_titles_desc['id']==narr]['description'].iloc[0]

        prompt=get_prompt_narr(narr_title, narr_desc)
        result=interact_with_model(prompt)
        print(result)
        new_row = pd.DataFrame([{'id':narr, 'description': result}])
        df_results = pd.concat([df_results, new_row], ignore_index=True)
        time.sleep(30)



100%|██████████| 96/96 [00:00<00:00, 4830.99it/s]


  0%|          | 0/21 [00:00<?, ?it/s]

**Description:** 

The subnarrative titled "Overpraising the West" explores the phenomenon of hyperbolic adulation directed towards Western nations, particularly in the realms of international relations, human rights, and military intervention. This narrative is set against a backdrop of global social, political, and economic dynamics, where the actions and policies of Western countries are often depicted as exemplary or morally superior compared to those of non-Western nations. Key actors in this narrative include political leaders, media outlets, and academic institutions that perpetuate the image of Western superiority. The narrative seeks to provoke a sense of pride and a belief in the benefits of Western hegemony, while simultaneously fostering skepticism or resentment towards other global players. Readers are encouraged to critically evaluate the implications of such overpraising, considering how it can lead to biased perceptions and foster divisions in international relations. T

 24%|██▍       | 5/21 [00:34<01:49,  6.87s/it]

**Description of the Subnarrative: Speculating War Outcomes**

In the context of ongoing global tensions and military conflicts, the subnarrative "Speculating War Outcomes" delves into the multifaceted implications of warfare, encompassing social, political, and economic dimensions. The narrative often features a range of actors, including government officials, military analysts, and international organizations, who contribute their perspectives on likely scenarios. It captures the uncertainty and fear surrounding the potential escalation of conflict, evoking emotions such as anxiety, urgency, and powerlessness among the public and stakeholders. The discourse aims to provoke critical thinking about the ramifications of war, urging readers to consider not only the immediate fallout but also long-term geopolitical shifts and humanitarian crises. Main textual characteristics include speculative language, the use of statistics or historical precedents to reinforce arguments, and a balanced

...

 90%|█████████ | 19/21 [07:21<01:04, 32.07s/it]

**Subnarrative Description: Amplifying Climate Fears**

This narrative unfolds within a broad social context marked by growing public awareness and concern over climate change. Politically, it engages various actors, including government officials, environmental organizations, scientists, and influential media outlets, all of whom play a pivotal role in shaping the discourse around climate fears. Economically, it touches upon the implications of climate change on industries, livelihoods, and global markets, often highlighting the urgency for sustainable practices and policies. Emotionally, the narrative is designed to provoke a sense of anxiety and urgency, instilling a fear of impending disasters such as natural catastrophes, resource scarcity, and societal upheaval. The underlying message encourages the reader to recognize the critical nature of climate challenges and to support advocacy for impactful climate action. Textually, this narrative utilizes vivid imagery and alarming stati

 95%|█████████▌| 20/21 [07:55<00:32, 32.66s/it]

**Subnarrative Description:**

In this narrative, the central theme revolves around the assertion that green policies—such as climate agreements, renewable energy initiatives, and biodiversity conservation strategies—are not merely expressions of environmental stewardship but rather strategic tools employed by nations to assert geopolitical power, influence international relations, and secure economic advantages. The social context encompasses a growing skepticism among the public regarding the sincerity of governmental and corporate commitments to sustainability, intertwining issues of climate justice with broader discussions about global inequalities. Politically, this narrative highlights key actors including powerful nations, international organizations, and influential corporations that may prioritize their own interests over genuine ecological concerns, thus raising questions about accountability and ethical leadership. Economically, it draws attention to how nations might levera

100%|██████████| 21/21 [08:29<00:00, 24.28s/it]


In [7]:
df_results

,id,description
0,URW5.2,The West Belongs on the Right Side of History*...
1,CC1.2,"The subnarrative titled ""Climate policies have..."
2,CC6.1,This subnarrative critiques the climate moveme...
3,URW1.1,This subnarrative seeks to reframe the conflic...
4,URW1.2,This subnarrative presents a perspective that ...
...,...,...
90,CC6.,**Description of the Subnarrative: Criticism o...
91,CC7.,**Narrative Title: Controversy about Green Tec...
92,CC8.,**Subnarrative Description:**\n\nThis narrativ...
93,CC9.,**Subnarrative Description: Amplifying Climate...


In [8]:
name='gpt_z_s_desc_title_description_complete'
df_results.to_pickle(f'/home/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}_v2.pkl')
df_results.to_json(f'/home/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}_v2.json', orient='records', indent=2, force_ascii=False)